# NutriScan — training the food recognition model

Run this notebook in **Google Colab** with a GPU: *Runtime → Change runtime type → T4 GPU*.

What it does:
1. Downloads the **Food-101** dataset and keeps the 20 dishes our app knows.
2. Downloads the **Central Asian Food Dataset (CAFD)** and takes 8 local dishes from it
   (plov, manty, baursak, beshbarmak, lagman, shashlik, samsa, doner).
3. Optionally adds our own photos (`kz_food.zip`) for dishes that no dataset has.
4. Fine-tunes a **ResNet-18** pretrained on ImageNet (transfer learning) on all 28 classes.
5. Shows loss/accuracy curves and a confusion matrix (put them on your slides).
6. Exports `food_model.pth` + `classes.json` → copy them to `backend/ml/weights/`.

⚠️ The model architecture and the test preprocessing here must stay the same as in `backend/ml/predictor.py`.

In [ ]:
import json, os, random, time, zipfile
from collections import Counter

import torch, torchvision
from PIL import Image
from torch import nn
from torch.utils.data import ConcatDataset, DataLoader, Dataset, WeightedRandomSampler
from torchvision import datasets, models, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', device)

## 1. Settings
These labels must match `Dish.ml_label` in the database (see `seed_demo.py`).
The order matters: `classes.json` is saved in this order and the backend reads it.

In [ ]:
# 20 classes taken from Food-101
FOOD101_CLASSES = [
    'pizza', 'hamburger', 'french_fries', 'hot_dog', 'sushi', 'ramen', 'fried_rice',
    'dumplings', 'spaghetti_bolognese', 'chicken_curry', 'steak', 'caesar_salad',
    'greek_salad', 'pancakes', 'omelette', 'waffles', 'ice_cream', 'chocolate_cake',
    'cheesecake', 'donuts',
]

# 8 classes taken from CAFD:  our class name -> CAFD folders that are merged into it.
# CAFD splits some dishes by variant (with/without soup, kind of meat); for calories one class is enough.
CAFD_CLASSES = {
    'plov': ['plov'],
    'manty': ['manty'],
    'baursak': ['bauyrsak'],
    'beshbarmak': ['beshbarmak-w-kazy', 'beshbarmak-wo-kazy'],
    'lagman': ['lagman-fried', 'lagman-w-soup', 'lagman-wo-soup'],
    'shashlik': ['shashlyk-chicken', 'shashlyk-chicken-v', 'shashlyk-kuskovoi', 'shashlyk-kuskovoi-v',
                 'shashlyk-minced-meat'],
    'samsa': ['samsa'],
    'doner': ['doner-lavash', 'doner-nan'],
}

# Optional: classes from our own photos in kz_food.zip (see section 3). Off by default.
USE_OWN_PHOTOS = False
OWN_CLASSES = ['blini']      # folder names inside kz_food.zip, used only when USE_OWN_PHOTOS = True

CUSTOM_CLASSES = list(CAFD_CLASSES) + (OWN_CLASSES if USE_OWN_PHOTOS else [])
CLASSES = FOOD101_CLASSES + CUSTOM_CLASSES   # 28 classes, Food-101 first

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 5          # 5 is enough for a demo; try 8-10 for better accuracy
LR = 1e-3
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # ImageNet values

MAX_TRAIN_PER_CLASS = 750   # cap for a CAFD class = size of a Food-101 class
MIN_IMAGES = 80             # warn if a class from our own photos has fewer photos than this
TEST_SHARE = 0.2            # our own photos: 20% are kept for testing
SEED = 42                   # fixed seed -> the same random choices on every run
print(len(CLASSES), 'classes')

## 2. Local dishes: Central Asian Food Dataset (CAFD)
Food-101 has no Kazakh dishes, so the 8 local classes come from **CAFD** by ISSAI, Nazarbayev
University: 16,499 photos of 42 Central Asian dishes, already split into `train/`, `val/`, `test/`.

- Dataset: https://huggingface.co/datasets/issai/Central_Asian_Food_Dataset
- Paper: Karabay, A., Bolatov, A., Varol, H. A., & Chan, M.-Y. (2023). *A Central Asian Food Dataset
  for Personalized Dietary Interventions.* Nutrients, 15(7), 1728. https://doi.org/10.3390/nu15071728
- License: **CC BY-NC 4.0** — free to use with attribution, **non-commercial only**. Our project is
  a student project, so this is fine; the trained model must not be used commercially, and the
  paper must be cited on the slides.

We use CAFD's own split: `train/` for training, `val/` + `test/` together for testing.
Only the folders listed in `CAFD_CLASSES` are downloaded.

In [ ]:
from huggingface_hub import snapshot_download

# Download only the folders we need (much faster than all 42 classes)
cafd_folders = [folder for folders in CAFD_CLASSES.values() for folder in folders]
snapshot_download(
    repo_id='issai/Central_Asian_Food_Dataset', repo_type='dataset', local_dir='cafd',
    allow_patterns=[f'{split}/{folder}/*' for split in ['train', 'val', 'test'] for folder in cafd_folders],
    ignore_patterns=['__MACOSX/*', '*.DS_Store'],   # macOS junk that was uploaded with the dataset
)

In [ ]:
def is_good_image(path):
    """True if PIL can fully read the file. Skips broken files and formats PIL cannot open."""
    try:
        with Image.open(path) as img:
            img.convert('RGB')
        return True
    except Exception:
        return False

def good_images(folder):
    """Readable images in a folder (sorted, so the result is the same on every run) + how many were skipped."""
    names = sorted(n for n in os.listdir(folder) if not n.startswith('.')) if os.path.isdir(folder) else []
    paths = [os.path.join(folder, n) for n in names]
    good = [p for p in paths if is_good_image(p)]
    return good, len(paths) - len(good)

custom_train_items, custom_test_items = [], []   # lists of (path, label) for all custom classes

# Opening every photo to check it takes 1-2 minutes
for name, folders in CAFD_CLASSES.items():
    train_paths, test_paths, skipped = [], [], 0
    for folder in folders:                        # merge several CAFD folders into one class
        for split, target in [('train', train_paths), ('val', test_paths), ('test', test_paths)]:
            good, bad = good_images(os.path.join('cafd', split, folder))
            target += good
            skipped += bad
    assert train_paths and test_paths, f'No CAFD images for class "{name}" - check the folder names in CAFD_CLASSES'

    found = len(train_paths)
    if found > MAX_TRAIN_PER_CLASS:               # random sample with a fixed seed
        train_paths = random.Random(SEED).sample(train_paths, MAX_TRAIN_PER_CLASS)
    label = CLASSES.index(name)
    custom_train_items += [(p, label) for p in train_paths]
    custom_test_items += [(p, label) for p in test_paths]
    print(f'{name:12s} train {len(train_paths):4d} (found {found:4d}) | test {len(test_paths):4d} | skipped {skipped} broken')

## 3. Optional: our own photos (`kz_food.zip`)
Skip this section unless you want a dish that neither dataset has (for example thin **blini**).
To use it: set `USE_OWN_PHOTOS = True` and list the dishes in `OWN_CLASSES` in section 1, then
**run the notebook again from the top**. The zip must look like this (folder names = `OWN_CLASSES`,
file names do not matter, `.jpg` / `.png`, about 100–150 photos per dish):

```
kz_food/
  blini/       img001.jpg, img002.jpg, ...
```

The cell asks you to upload `kz_food.zip` (or set `USE_DRIVE = True` and put the zip in the root of
*My Drive*), skips broken files and splits each class **80% train / 20% test**.
With `USE_OWN_PHOTOS = False` the cell does nothing.

In [ ]:
USE_DRIVE = False                          # True = take the zip from Google Drive instead of uploading

if USE_OWN_PHOTOS:
    ZIP_PATH = 'kz_food.zip'
    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        ZIP_PATH = '/content/drive/MyDrive/kz_food.zip'
    elif not os.path.exists(ZIP_PATH):
        from google.colab import files
        files.upload()                     # choose kz_food.zip in the dialog

    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall('kz_unzipped')

    # Find the folder that holds the class folders (works with or without a top-level kz_food/ in the zip)
    own_dir = None
    for root, dirs, _ in os.walk('kz_unzipped'):
        if '__MACOSX' not in root and any(c in dirs for c in OWN_CLASSES):
            own_dir = root
            break
    assert own_dir, 'No class folders found in the zip. Expected kz_food/<class_name>/... for every name in OWN_CLASSES'

    for name in OWN_CLASSES:
        good, skipped = good_images(os.path.join(own_dir, name))
        assert len(good) >= 2, f'Not enough images for class "{name}" - add photos or remove it from OWN_CLASSES'

        random.Random(SEED).shuffle(good)  # fixed seed -> same split every time
        n_test = max(1, round(len(good) * TEST_SHARE))
        label = CLASSES.index(name)
        custom_test_items += [(p, label) for p in good[:n_test]]
        custom_train_items += [(p, label) for p in good[n_test:]]

        warning = f'   <-- WARNING: fewer than {MIN_IMAGES} images, collect more' if len(good) < MIN_IMAGES else ''
        print(f'{name:12s} {len(good):4d} images = {len(good) - n_test} train + {n_test} test, '
              f'skipped {skipped} broken{warning}')
else:
    print('Own photos are off - only Food-101 and CAFD are used.')

## 4. Data
Food-101 is ~5 GB, the first download takes a few minutes.

**Class imbalance.** Each Food-101 class has 750 train photos. CAFD classes are capped at 750, but
some have fewer (see the counts printed above), and a class from our own photos has only ~100.
If we trained as usual, the model would see a small class less often and would learn to prefer the
big ones. We use a `WeightedRandomSampler`: every image gets the weight
`1 / (number of images in its class)`, so **each class is picked equally often**. Photos of the
small classes are simply repeated more often, and the random augmentations (crop, flip, colors)
make every repeat look a bit different. We chose this over class weights in the loss because it
is easy to explain and every batch contains a mix of all classes.
The test set is **not** rebalanced — it must show real accuracy.

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
# Same as backend/ml/predictor.py - do not change one without the other
test_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class SelectedFood(Dataset):
    """Food-101 limited to FOOD101_CLASSES, with labels renumbered 0..19 (their position in CLASSES)."""
    def __init__(self, split, transform):
        self.base = datasets.Food101('data', split=split, download=True)
        self.transform = transform
        old_to_new = {self.base.classes.index(c): i for i, c in enumerate(FOOD101_CLASSES)}
        self.items = [(idx, old_to_new[lbl]) for idx, lbl in enumerate(self.base._labels) if lbl in old_to_new]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        idx, label = self.items[i]
        image, _ = self.base[idx]
        return self.transform(image), label

class CustomFood(Dataset):
    """Custom classes (CAFD or our own photos): a list of (path, label), labels 20 and up."""
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        return self.transform(Image.open(path).convert('RGB')), label

# One train dataset and one test dataset from both sources
food_train = SelectedFood('train', train_tf)
food_test = SelectedFood('test', test_tf)
train_ds = ConcatDataset([food_train, CustomFood(custom_train_items, train_tf)])
test_ds = ConcatDataset([food_test, CustomFood(custom_test_items, test_tf)])

# Balance the classes: weight of an image = 1 / size of its class (see the text above).
# train_labels is in the same order as train_ds: Food-101 first, then the custom photos.
train_labels = [label for _, label in food_train.items] + [label for _, label in custom_train_items]
class_counts = Counter(train_labels)
sample_weights = [1.0 / class_counts[label] for label in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_labels), replacement=True)

# The sampler decides the order, so shuffle=True is not needed here
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
print('train images:', len(train_ds), '| test images:', len(test_ds))
print('train images per class:', {CLASSES[i]: class_counts[i] for i in range(len(CLASSES))})

## 5. Model: pretrained ResNet-18 with a new last layer

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))  # same as predictor.py
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## 6. Training

In [ ]:
def evaluate():
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for x, y in test_dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += criterion(out, y).item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return loss_sum / total, correct / total

history = {'train_loss': [], 'test_loss': [], 'test_acc': []}
for epoch in range(EPOCHS):
    model.train()
    start, loss_sum, n = time.time(), 0.0, 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(y)
        n += len(y)
    scheduler.step()
    test_loss, test_acc = evaluate()
    history['train_loss'].append(loss_sum / n)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    print(f'epoch {epoch + 1}/{EPOCHS}  train_loss {loss_sum / n:.3f}  test_loss {test_loss:.3f}  test_acc {test_acc:.1%}  ({time.time() - start:.0f}s)')

## 7. Results for the slides

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history['train_loss'], label='train'); ax[0].plot(history['test_loss'], label='test')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(history['test_acc']); ax[1].set_title('Test accuracy'); ax[1].set_xlabel('epoch')
plt.tight_layout(); plt.savefig('training_curves.png', dpi=150); plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
model.eval()
preds, labels = [], []
with torch.no_grad():
    for x, y in test_dl:
        preds += model(x.to(device)).argmax(1).cpu().tolist()
        labels += y.tolist()

all_ids = list(range(len(CLASSES)))
print(classification_report(labels, preds, labels=all_ids, target_names=CLASSES, digits=3, zero_division=0))

# Accuracy separately for Food-101 and for the custom classes (labels 20 and up).
# The overall number is dominated by Food-101 (250 test photos per class), so we look at the custom ones on their own.
n_food = len(FOOD101_CLASSES)
food_hits = [p == l for p, l in zip(preds, labels) if l < n_food]
custom_hits = [p == l for p, l in zip(preds, labels) if l >= n_food]
print(f'Food-101 classes: {sum(food_hits) / len(food_hits):.1%} accuracy on {len(food_hits)} test images')
print(f'Custom classes:   {sum(custom_hits) / len(custom_hits):.1%} accuracy on {len(custom_hits)} test images')
for i, name in enumerate(CUSTOM_CLASSES, start=n_food):
    hits = [p == l for p, l in zip(preds, labels) if l == i]
    print(f'  {name:12s} {sum(hits)}/{len(hits)} correct')

# Big figure so that 28 labels are readable
fig, ax = plt.subplots(figsize=(18, 18))
ConfusionMatrixDisplay.from_predictions(labels, preds, labels=all_ids, display_labels=CLASSES,
                                        xticks_rotation=90, ax=ax, colorbar=False)
ax.tick_params(labelsize=11)
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=150); plt.show()

## 8. Export for the Django backend
Download the files and put `food_model.pth` and `classes.json` in `backend/ml/weights/`
(replace the old ones), then restart Django. `classes.json` now has 28 names.

In [ ]:
model.cpu()
torch.save(model.state_dict(), 'food_model.pth')
with open('classes.json', 'w') as f:
    json.dump(CLASSES, f)

from google.colab import files
for name in ['food_model.pth', 'classes.json', 'training_curves.png', 'confusion_matrix.png']:
    files.download(name)